# Mask R-CNN training — book & card segmentation

Runs the reproducible training pipeline from the repository on a Colab GPU.

**Before running:** Runtime → Change runtime type → **T4 GPU**. Then Runtime → Run all.

All logic lives in `models/` — this notebook only clones the repo, runs the scripts and packages the results.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Get the code and data

In [ ]:
REPO = "https://github.com/abdullahks-devhub/xis-cv-assessment.git"
BRANCH = "feature/training"

!rm -rf xis-cv-assessment
!git clone -q --depth 1 -b {BRANCH} {REPO}
%cd xis-cv-assessment
!pip install -q pycocotools pyyaml
!ls dataset/splits

## 2. Train
Configuration: `models/config.yaml`. Logs train/val loss and val mAP every epoch; keeps the best weights by val mask mAP@0.5:0.95.

In [ ]:
!python -m models.train --config models/config.yaml

## 3. Evaluate on the held-out test set (and val)

In [ ]:
!python -m models.evaluate --weights models/output/best.pth --split test
!python -m models.evaluate --weights models/output/best.pth --split val

## 4. Results

In [ ]:
from IPython.display import Image, display
display(Image("models/output/curves.png"))
display(Image("inference/test_predictions/grid.jpg", width=1100))

## 5. Download results
Downloads `training_results.zip` (weights, history, curves, metrics, prediction overlays). Unzip it into the repository root on your computer.

In [ ]:
!zip -q -r training_results.zip models/output inference/test_predictions inference/val_predictions
from google.colab import files
files.download("training_results.zip")